<a href="https://colab.research.google.com/github/Bartimeyss/Ml-contest/blob/main/ml_kontest_yo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"嗨，毫升"

# Import modules and data

In [ ]:
%pip install torch
%pip install optuna

import numpy as np
import pandas as pd
import optuna
import torch as trc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 15.1 MB/s eta 0:00:00


In [ ]:

import os

#download files

os.environ['KAGGLE_USERNAME'] = "nezeritka"
os.environ['KAGGLE_KEY'] = "4da40b42a8f49222ac788d2832884481"
!kaggle competitions download -c scooter-price-prediction-edu-2026
!unzip scooter-price-prediction-edu-2026.zip
!rm scooter-price-prediction-edu-2026.zip



100% 159k/159k [00:00<00:00, 103MB/s]

Archive:  scooter-price-prediction-edu-2026.zip
  inflating: sample_submission.csv   
  inflating: test.csv                
  inflating: train.csv               


In [ ]:
# Проверка загрузки

train_path = f"train.csv"
test_path = f"test.csv"
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)




In [ ]:
#train
print(test.head())
print(train.head())

   id  trip_duration_min  distance_km  battery_level_start  temperature_c  \
0   0          11.328001     2.984336            55.779495       8.091798   
1   1          38.498660    11.646348            82.460577      14.587610   
2   2          12.336503     3.903070            27.184917      26.280163   
3   3          18.232429     5.046844            25.686523            NaN   
4   4          22.992648     8.149461            39.617766       9.642400   

   wind_speed  demand_index  distance_km_noisy city_zone scooter_model  \
0    4.066090      0.663601           1.963757    center             B   
1    0.324484      0.025097          10.144005    suburb             B   
2    0.803103      0.550207           0.263852      park             A   
3    9.745415      0.365754           7.439801    center             B   
4    4.688548      0.991382           6.484963    suburb             B   

   is_weekend  avg_price_last_week  route_complexity  driver_experience  \
0           0    

# Process data



In [ ]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer


train = train.drop(columns=['id'], errors='ignore')
test = test.drop(columns=['id'], errors='ignore')
y = train['rental_price']
X = train.drop(columns=['rental_price'])

categorical_columns = ['city_zone', 'scooter_model']
numerical_columns = X.select_dtypes(include=['int64', 'float64']).columns

imputer = SimpleImputer(strategy='median')
X[numerical_columns] = imputer.fit_transform(X[numerical_columns])
test[numerical_columns] = imputer.transform(test[numerical_columns])

encoder = OneHotEncoder(sparse_output=False)

X_cat = encoder.fit_transform(X[categorical_columns])
test_cat = encoder.transform(test[categorical_columns])

cat_feature_names = encoder.get_feature_names_out(categorical_columns)
X_cat = pd.DataFrame(X_cat, columns=cat_feature_names, index=X.index)
test_cat = pd.DataFrame(test_cat, columns=cat_feature_names, index=test.index)

X = X.drop(columns=categorical_columns)
test = test.drop(columns=categorical_columns)

X = pd.concat([X, X_cat], axis=1)
test = pd.concat([test, test_cat], axis=1)

print(X)
print(test)

      trip_duration_min  distance_km  battery_level_start  temperature_c  \
0             25.599707     7.441135            78.519717       6.128784   
1             57.289287     8.770495            58.708818      15.947497   
2             45.259667     6.147492            31.714901      26.099837   
3             37.926217     7.740660            86.634377      11.560209   
4             13.581025     3.846835            63.223723      15.542815   
...                 ...          ...                  ...            ...   
1195          52.934524    12.453208            29.536835      16.305378   
1196          58.541893    10.690922            22.916884       8.569641   
1197          58.288282     9.855832            24.219516      17.324568   
1198          46.230851    12.539495            86.733129      28.792560   
1199          12.154743     4.280519            29.437799      12.993021   

      wind_speed  demand_index  distance_km_noisy  is_weekend  \
0       8.680982      

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

tsne = TSNE(n_components=2, learning_rate="auto", random_state=69420)
X_tsne = tsne.fit_transform(X_scaled)

plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y, cmap='viridis')
plt.colorbar(label='rental_price')
plt.show()

KeyboardInterrupt: 

In [ ]:
for col in numerical_columns:
    print(col, X[col].quantile([0.01, 0.99]))

# Gridsearch or optuna ig

In [ ]:
%pip install catboost

from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.9 MB/s eta 0:00:00


In [ ]:
features = ['id', 'trip_duration_min', 'distance_km', 'battery_level_start',
       'temperature_c', 'wind_speed', 'demand_index', 'distance_km_noisy', 'is_weekend',
       'avg_price_last_week', 'route_complexity', 'driver_experience',
       'weather_rating']

target = 'rental_price'

X_train, X_test, y_train, y_test = train_test_split(train[features], train[target], test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

NameError: name 'train_test_split' is not defined

In [ ]:
study = optuna.create_study(direction="maximize")

def objective(trial):
    param = {
        'verbose': False,
        "n_estimators": trial.suggest_int("n_estimators", 100, 10000),
        "max_depth": trial.suggest_int("max_depth", 2, 16),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.4),
        "task_type": "GPU"
    }
    model = CatBoostRegressor(**param)
    model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)
    preds = model.predict(X_test)
    return r2_score(y_test, preds)

study.optimize(objective, n_trials=100)
print(study.best_value)


[I 2026-04-13 12:54:52,795] A new study created in memory with name: no-name-6622a677-d35f-40b4-b797-aedac6c6f5af
[W 2026-04-13 12:54:52,800] Trial 0 failed with parameters: {'n_estimators': 3772, 'max_depth': 13, 'learning_rate': 0.0936758174046596} because of the following error: NameError("name 'CatBoostRegressor' is not defined").
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipykernel_3208/3166736875.py", line 11, in objective
    model = CatBoostRegressor(**param)
            ^^^^^^^^^^^^^^^^^
NameError: name 'CatBoostRegressor' is not defined
[W 2026-04-13 12:54:52,808] Trial 0 failed with value None.


NameError: name 'CatBoostRegressor' is not defined

In [ ]:


features = ['id', 'trip_duration_min', 'distance_km', 'battery_level_start',
       'temperature_c', 'wind_speed', 'demand_index', 'distance_km_noisy', 'is_weekend',
       'avg_price_last_week', 'route_complexity', 'driver_experience',
       'weather_rating']

target = 'rental_price'

ids = train['id']
X_train, X_test, y_train, y_test, ids_train, ids_test = train_test_split(train[features], train[target], ids, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)
# Инициализация и обучение модели
model = CatBoostRegressor(
    iterations=2000,
    learning_rate=0.1,
    depth=8,
    verbose=100,
    loss_function='RMSE'
)

model.fit(X_train, y_train, eval_set=(X_val, y_val), early_stopping_rounds=50)

print("Обучение завершено.")

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
r2_score(y_pred, y_test)